### Import libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

### 1. Load cleaned dataset

In [11]:

DATA_PATH = Path("../data/processed/crop_yield_cleaned.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

required_columns = {
    "crop_type",
    "rainfall",
    "temperature",
    "fertilizer",
    "nitrogen",
    "phosphorus",
    "potassium",
    "yield",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise KeyError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

print("All required columns are available.")

Dataset shape: (999, 8)
Columns: ['rainfall', 'fertilizer', 'temperature', 'nitrogen', 'phosphorus', 'potassium', 'yield', 'crop_type']
All required columns are available.


### 2. Feature Engineering (Domain-Specific Transformations)

### Create selected derived variables

In [12]:
import numpy as np
import pandas as pd


def create_crop_features(data: pd.DataFrame) -> pd.DataFrame:
    """Create row-level derived features for crop-yield modelling."""
    df_feat = data.copy()

    df_feat["total_npk"] = (
        df_feat["nitrogen"]
        + df_feat["phosphorus"]
        + df_feat["potassium"]
    )

    total_npk_safe = df_feat["total_npk"].replace(0, np.nan)

    df_feat["n_proportion"] = (
        df_feat["nitrogen"] / total_npk_safe
    )

    df_feat["p_proportion"] = (
        df_feat["phosphorus"] / total_npk_safe
    )

    df_feat["temperature_squared"] = (
        df_feat["temperature"] ** 2
    )

    df_feat["rainfall_fertilizer_interaction"] = (
        df_feat["rainfall"]
        * df_feat["fertilizer"]
    )

    return df_feat

In [13]:
df_engineered = create_crop_features(df)

print("Original shape:", df.shape)
print("Engineered shape:", df_engineered.shape)

df_engineered.head()

Original shape: (999, 8)
Engineered shape: (999, 13)


,rainfall,fertilizer,temperature,nitrogen,phosphorus,potassium,yield,crop_type,total_npk,n_proportion,p_proportion,temperature_squared,rainfall_fertilizer_interaction
0,1230,80,28,80,24,20,12.0,Corn,124,0.645161,0.193548,784,98400
1,480,60,36,70,20,18,8.0,Sorghum,108,0.648148,0.185185,1296,28800
2,1250,75,29,78,22,19,11.0,Soybean,119,0.655462,0.184874,841,93750
3,450,65,35,70,19,18,9.0,Sorghum,107,0.654206,0.177570,1225,29250
4,1200,80,27,79,22,19,11.0,Wheat,120,0.658333,0.183333,729,96000


### Confirm new features

In [18]:
new_features = [
    column
    for column in df_engineered.columns
    if column not in df.columns
]

print("Created features:", new_features)

Created features: ['total_npk', 'n_proportion', 'p_proportion', 'temperature_squared', 'rainfall_fertilizer_interaction']


In [19]:

engineered_columns = new_features

df_engineered[engineered_columns] = (
    df_engineered[engineered_columns]
    .replace([np.inf, -np.inf], np.nan)
)

quality_check = pd.DataFrame({
    "missing_values": df_engineered[engineered_columns].isna().sum(),
    "minimum": df_engineered[engineered_columns].min(),
    "maximum": df_engineered[engineered_columns].max()
})

quality_check

,missing_values,minimum,maximum
total_npk,0,87.00000,139.000000
n_proportion,0,0.53913,0.774775
p_proportion,0,0.12000,0.287129
temperature_squared,0,400.00000,1600.000000
rainfall_fertilizer_interaction,0,20000.00000,136416.000000


### Review summary statistics

In [20]:
engineered_features = [
    "total_npk",
    "n_proportion",
    "p_proportion",
    "temperature_squared",
    "rainfall_fertilizer_interaction"
]

df_engineered[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
total_npk,999.0,113.227227,10.671231,87.00000,105.000000,114.000000,121.000000,139.000000
n_proportion,999.0,0.657811,0.042597,0.53913,0.629470,0.655738,0.686441,0.774775
p_proportion,999.0,0.192764,0.033126,0.12000,0.166667,0.193548,0.217742,0.287129
temperature_squared,999.0,908.563564,344.392221,400.00000,625.000000,841.000000,1156.000000,1600.000000
rainfall_fertilizer_interaction,999.0,66710.889890,26741.401694,20000.00000,45128.000000,65162.000000,88016.000000,136416.000000


### Save the engineered dataset

In [ ]:
from pathlib import Path

OUTPUT_PATH = Path(
    "../data/processed/crop_yield_engineered.csv"
)

df_engineered.to_csv(OUTPUT_PATH, index=False)

print(f"Engineered dataset saved to: {OUTPUT_PATH}")

Engineered dataset saved to: ../data/processed/crop_yield_engineered.csv


## Breakdown of Key Engineered Features

| Feature Category           | Engineered Column                 | Mathematical Formula                       | Agronomic Significance                                                                                            |
| -------------------------- | --------------------------------- | ------------------------------------------ | ----------------------------------------------------------------------------------------------------------------- |
| **Total Nutrient Level**   | `total_npk`                       | $N + P + K$                                | Summarises the combined recorded levels of nitrogen, phosphorus and potassium available for crop growth.          |
| **Nitrogen Composition**   | `n_proportion`                    | $\frac{N}{N+P+K}$                          | Measures the proportion of total NPK contributed by nitrogen and helps describe nutrient balance.                 |
| **Phosphorus Composition** | `p_proportion`                    | $\frac{P}{N+P+K}$                          | Measures the proportion of total NPK contributed by phosphorus and helps describe nutrient balance.               |
| **Thermal Response**       | `temperature_squared`             | $\text{Temperature}^2$                     | Allows the model to capture a possible non-linear relationship between temperature and crop yield.                |
| **Input Synergy**          | `rainfall_fertilizer_interaction` | $\text{Rainfall} \times \text{Fertilizer}$ | Captures how the relationship between fertiliser use and crop yield may vary under different rainfall conditions. |
